# Smart Cargo — Análisis Exploratorio de Datos (Fase 1)

Este notebook cubre la **Fase 1** del proyecto: carga, limpieza y análisis
exploratorio del dataset de recursos humanos orientado a predecir la deserción
de empleados.

La lógica de limpieza y de features vive en `src/`, de modo que este notebook
solo la consume. Así el análisis es reproducible y el código de producción no
queda duplicado en celdas.

**Contenido**
1. Setup e imports
2. Carga y limpieza de datos (`src.data_processing`)
3. Estadísticas descriptivas y análisis del target
4. Visualizaciones del EDA
5. Matriz de correlación

In [ ]:
import sys
from pathlib import Path

# Permite importar `src` cuando el notebook se ejecuta desde notebooks/
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import DEFAULT_CONFIG, DEFAULT_RAW_PATH, FIGURES_DIR
from src.data_processing import clean_dataset, load_dataset, missing_value_report

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

pd.set_option('display.max_columns', 100)

## 1. Carga y limpieza

Coloca el dataset en `data/raw/base_proyecto_.xlsx` (o ajusta `DATA_PATH`).
La limpieza —estandarización de categóricas e imputación de `sueldo`, `area` y
`edad`— se delega a `src.data_processing.clean_dataset`.

In [ ]:
DATA_PATH = DEFAULT_RAW_PATH  # cambia esta ruta si usas otro archivo

df_raw = load_dataset(DATA_PATH)
print(f'Dataset original: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas')
df_raw.head()

In [ ]:
print('Valores nulos antes de la limpieza:')
print(missing_value_report(df_raw) or 'Sin valores nulos')

In [ ]:
df = clean_dataset(df_raw, DEFAULT_CONFIG)
print(f'Dataset limpio: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print('Valores nulos después de la limpieza:')
print(missing_value_report(df) or 'Sin valores nulos')

## 2. Estadísticas descriptivas y análisis del target

In [ ]:
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != 'id']
print(f'DIMENSIONES: {df.shape[0]:,} registros | {df.shape[1]} variables')
df[numeric_cols].describe().T

In [ ]:
target_counts = df['DESERCION'].value_counts()
target_pct = df['DESERCION'].value_counts(normalize=True) * 100
print(f'ACTIVOS (0):    {target_counts[0]:,} ({target_pct[0]:.2f}%)')
print(f'DESERCIÓN (1):  {target_counts[1]:,} ({target_pct[1]:.2f}%)')
print(f'Ratio de desbalance: {target_counts[0]/target_counts[1]:.2f}:1')

In [ ]:
tasa_empresa = df.groupby('empresa_gestion')['DESERCION'].agg(['count','sum','mean']).round(4)
tasa_empresa.columns = ['Total', 'Deserciones', 'Tasa']
tasa_empresa.sort_values('Tasa', ascending=False)

### Outliers (método IQR)

In [ ]:
for var in ['sueldo', 'dias_trabajo', 'edad']:
    if var in df.columns:
        Q1, Q3 = df[var].quantile(0.25), df[var].quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        outliers = df[(df[var] < lower) | (df[var] > upper)]
        print(f'{var}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)')

## 3. Visualizaciones del EDA

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('FASE 1: ANÁLISIS EXPLORATORIO DE DATOS (EDA)', fontsize=16, fontweight='bold')

colors = ['#2ecc71', '#e74c3c']

# 1. Distribución del target
axes[0, 0].pie(target_counts.values, labels=['Activo (0)', 'Deserción (1)'],
               autopct='%1.1f%%', colors=colors, startangle=90, explode=(0, 0.1))
axes[0, 0].set_title('Distribución del Target')

# 2. Sueldo por deserción
df.boxplot(column='sueldo', by='DESERCION', ax=axes[0, 1])
axes[0, 1].set_title('Distribución de Sueldo por Deserción')
axes[0, 1].set_xlabel('DESERCION')

# 3. Tasa de deserción por empresa
tasa_plot = tasa_empresa.sort_values('Tasa', ascending=True)
axes[0, 2].barh(range(len(tasa_plot)), tasa_plot['Tasa'] * 100, color='steelblue')
axes[0, 2].set_yticks(range(len(tasa_plot)))
axes[0, 2].set_yticklabels([n[:15] for n in tasa_plot.index], fontsize=8)
axes[0, 2].set_xlabel('Tasa de Deserción (%)')
axes[0, 2].set_title('Tasa de Deserción por Empresa')

# 4. Clima laboral vs deserción
clima = df.groupby(['clima(1-10)', 'DESERCION']).size().unstack(fill_value=0)
clima.div(clima.sum(axis=1), axis=0).mul(100).plot(kind='bar', stacked=True, ax=axes[1, 0], color=colors)
axes[1, 0].set_title('Clima Laboral vs Deserción')
axes[1, 0].legend(['Activo', 'Deserción'])

# 5. Estrés vs deserción
estres = df.groupby(['estres(1-3)', 'DESERCION']).size().unstack(fill_value=0)
estres.div(estres.sum(axis=1), axis=0).mul(100).plot(kind='bar', stacked=True, ax=axes[1, 1], color=colors)
axes[1, 1].set_title('Nivel de Estrés vs Deserción')
axes[1, 1].legend(['Activo', 'Deserción'])

# 6. Ausentismo
df[df['DESERCION'] == 0]['dias_ausentismo'].hist(bins=15, alpha=0.5, label='Activo', ax=axes[1, 2], color='green')
df[df['DESERCION'] == 1]['dias_ausentismo'].hist(bins=15, alpha=0.5, label='Deserción', ax=axes[1, 2], color='red')
axes[1, 2].set_title('Distribución de Días de Ausentismo')
axes[1, 2].legend()

plt.tight_layout()
plt.show()

## 4. Matriz de correlación

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
corr_matrix = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=ax)
ax.set_title('MATRIZ DE CORRELACIÓN', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Conclusiones

- El target está fuertemente desbalanceado, por lo que **PR-AUC** es la métrica
  principal y se justifica el uso de SMOTE.
- `clima(1-10)`, `estres(1-3)` y `dias_ausentismo` muestran separación entre
  activos y desertores, lo que respalda su uso como features.
- La tasa de deserción varía entre empresas de gestión: conviene incluirla como
  variable categórica (target encoding en la Fase 2).

El resto del pipeline (features, modelado y evaluación) vive en `src/` y se
ejecuta con `python -m src.main`.